# 🌸 Klasifikasi Bunga Iris dengan KNN & Random Forest
**Study Club Data Science Beginner — Tugas 1**

---

### 📌 Informasi Proyek
- **Dataset:** Iris Flower Dataset (Kaggle)
- **Algoritma:** K-Nearest Neighbors (KNN) & Random Forest (Ensemble Learning)
- **Tujuan:** Mengklasifikasikan spesies bunga iris (Setosa, Versicolor, Virginica) berdasarkan fitur pengukuran kelopak dan mahkota bunga.

---

### 📋 ML Workflow
1. Data Collecting
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing
4. Model Training
5. Model Evaluation

---
## 📦 Import Library
Mengimpor semua library yang dibutuhkan untuk analisis data dan pembuatan model.

In [6]:
# Library dasar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Library Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

# Atur style visualisasi
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 6)

print('✅ Semua library berhasil diimpor!')

ModuleNotFoundError: No module named 'pandas'

---
## 1️⃣ Data Collecting

Dataset yang digunakan adalah **Iris Flower Dataset** dari Kaggle. Dataset ini berisi 150 sampel bunga iris dengan 5 fitur utama:

| Kolom | Keterangan |
|---|---|
| `SepalLengthCm` | Panjang sepal (kelopak luar) dalam cm |
| `SepalWidthCm` | Lebar sepal dalam cm |
| `PetalLengthCm` | Panjang petal (mahkota) dalam cm |
| `PetalWidthCm` | Lebar petal dalam cm |
| `Species` | Label kelas: Iris-setosa, Iris-versicolor, Iris-virginica |

In [ ]:
# Load dataset
# Jika menggunakan Google Colab, upload file Iris.csv terlebih dahulu
df = pd.read_csv('Iris.csv')

# Hapus kolom 'Id' karena tidak relevan untuk analisis
df.drop(columns=['Id'], inplace=True)

print('✅ Dataset berhasil dimuat!')
print(f'📊 Jumlah baris: {df.shape[0]}, Jumlah kolom: {df.shape[1]}')
df.head(10)

---
## 2️⃣ Exploratory Data Analysis (EDA)

Pada tahap ini kita akan memahami struktur, distribusi, dan pola dari dataset sebelum melakukan pemodelan.

In [ ]:
# === 2.1 Informasi Umum Dataset ===
print('=' * 50)
print('📋 INFO DATASET')
print('=' * 50)
df.info()

In [ ]:
# === 2.2 Statistik Deskriptif ===
print('📊 Statistik Deskriptif:')
df.describe().round(2)

In [ ]:
# === 2.3 Cek Missing Values ===
print('🔍 Cek Missing Values:')
missing = df.isnull().sum()
print(missing)
print(f'\n✅ Total missing values: {missing.sum()} — Dataset bersih!')

In [ ]:
# === 2.4 Distribusi Kelas Target ===
print('🌸 Distribusi Spesies:')
print(df['Species'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
df['Species'].value_counts().plot(kind='bar', ax=axes[0], color=sns.color_palette('Set2', 3), edgecolor='black')
axes[0].set_title('Jumlah Sampel per Spesies', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Spesies')
axes[0].set_ylabel('Jumlah')
axes[0].tick_params(axis='x', rotation=15)

# Pie chart
df['Species'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                                   colors=sns.color_palette('Set2', 3), startangle=90)
axes[1].set_title('Proporsi Spesies', fontsize=13, fontweight='bold')
axes[1].set_ylabel('')

plt.suptitle('Distribusi Kelas Target (Species)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('✅ Dataset seimbang — masing-masing spesies berjumlah 50 sampel (33.3%).')

In [ ]:
# === 2.5 Distribusi Fitur (Histogram) ===
features = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(features):
    for species in df['Species'].unique():
        subset = df[df['Species'] == species][col]
        axes[i].hist(subset, alpha=0.6, label=species, bins=15, edgecolor='white')
    axes[i].set_title(f'Distribusi {col}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frekuensi')
    axes[i].legend(fontsize=9)

plt.suptitle('Distribusi Fitur per Spesies', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# === 2.6 Boxplot per Fitur ===
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(features):
    sns.boxplot(data=df, x='Species', y=col, ax=axes[i], palette='Set2')
    axes[i].set_title(f'Boxplot: {col}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Spesies')
    axes[i].set_ylabel(col)

plt.suptitle('Boxplot Fitur per Spesies', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# === 2.7 Heatmap Korelasi ===
plt.figure(figsize=(9, 7))
corr = df[features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, linewidths=0.5, vmin=-1, vmax=1,
            annot_kws={'size': 12})
plt.title('Heatmap Korelasi Antar Fitur', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('💡 PetalLengthCm dan PetalWidthCm memiliki korelasi sangat tinggi (0.96).')

In [ ]:
# === 2.8 Pairplot ===
print('📊 Membuat Pairplot... (mungkin perlu beberapa detik)')
pairplot = sns.pairplot(df, hue='Species', palette='Set2', diag_kind='kde',
                         plot_kws={'alpha': 0.7})
pairplot.fig.suptitle('Pairplot Fitur Iris Dataset', fontsize=15, fontweight='bold', y=1.02)
plt.show()
print('💡 Iris-setosa sangat mudah dipisahkan. Versicolor dan Virginica sedikit overlap.')

---
## 3️⃣ Data Preprocessing

Sebelum melatih model, kita perlu mempersiapkan data:
1. **Encoding Label** — Mengubah target (Species) dari teks menjadi angka
2. **Feature & Target Split** — Memisahkan fitur (X) dan target (y)
3. **Train-Test Split** — Membagi data menjadi data latih (80%) dan data uji (20%)
4. **Feature Scaling** — Standarisasi fitur agar semua fitur berada pada skala yang sama (penting untuk KNN)

In [ ]:
# === 3.1 Label Encoding ===
le = LabelEncoder()
df['Species_encoded'] = le.fit_transform(df['Species'])

print('📌 Mapping Label Encoding:')
for idx, cls in enumerate(le.classes_):
    print(f'  {idx} → {cls}')

In [ ]:
# === 3.2 Pisahkan Fitur (X) dan Target (y) ===
X = df[features]
y = df['Species_encoded']

print(f'✅ Shape fitur X: {X.shape}')
print(f'✅ Shape target y: {y.shape}')

In [ ]:
# === 3.3 Train-Test Split (80:20) ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'📊 Ukuran Data Latih  : {X_train.shape[0]} sampel')
print(f'📊 Ukuran Data Uji    : {X_test.shape[0]} sampel')
print(f'💡 Stratify=True memastikan proporsi kelas seimbang di train & test set.')

In [ ]:
# === 3.4 Feature Scaling (StandardScaler) ===
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Fit hanya pada data train!
X_test_scaled  = scaler.transform(X_test)        # Transform pada data test

print('✅ Feature Scaling selesai!')
print(f'   Mean fitur setelah scaling (train): {X_train_scaled.mean(axis=0).round(4)}')
print(f'   Std  fitur setelah scaling (train): {X_train_scaled.std(axis=0).round(4)}')

---
## 4️⃣ Model Training

Kita akan melatih **dua model** dan membandingkan performanya:
1. **K-Nearest Neighbors (KNN)** — Algoritma berbasis jarak, mengklasifikasikan data berdasarkan k tetangga terdekat
2. **Random Forest** — Algoritma Ensemble Learning yang menggabungkan banyak decision tree

In [ ]:
# === 4.1 Mencari K Optimal untuk KNN ===
k_values = range(1, 21)
cv_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

best_k = k_values[np.argmax(cv_scores)]

plt.figure(figsize=(10, 5))
plt.plot(k_values, cv_scores, marker='o', color='steelblue', linewidth=2, markersize=7)
plt.axvline(x=best_k, color='red', linestyle='--', label=f'K optimal = {best_k}')
plt.title('Cross-Validation Accuracy vs Nilai K (KNN)', fontsize=13, fontweight='bold')
plt.xlabel('Jumlah Tetangga (K)')
plt.ylabel('CV Accuracy')
plt.xticks(k_values)
plt.legend()
plt.tight_layout()
plt.show()
print(f'✅ K optimal ditemukan: K = {best_k} dengan CV Accuracy = {max(cv_scores):.4f}')

In [ ]:
# === 4.2 Training Model KNN ===
knn_model = KNeighborsClassifier(n_neighbors=best_k)
knn_model.fit(X_train_scaled, y_train)

print(f'✅ Model KNN (K={best_k}) berhasil dilatih!')

In [ ]:
# === 4.3 Training Model Random Forest ===
rf_model = RandomForestClassifier(
    n_estimators=100,   # Jumlah pohon keputusan
    max_depth=None,     # Kedalaman maksimal pohon (None = tidak dibatasi)
    random_state=42
)
rf_model.fit(X_train_scaled, y_train)

print('✅ Model Random Forest (100 trees) berhasil dilatih!')

In [ ]:
# === 4.4 Feature Importance (Random Forest) ===
importances = rf_model.feature_importances_
feat_imp = pd.Series(importances, index=features).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
colors = sns.color_palette('Set2', len(features))
feat_imp.plot(kind='barh', color=colors, edgecolor='black')
plt.title('Feature Importance — Random Forest', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()
print('💡 PetalLengthCm dan PetalWidthCm adalah fitur paling penting untuk klasifikasi.')

---
## 5️⃣ Model Evaluation

Evaluasi performa kedua model menggunakan:
- **Accuracy** — Proporsi prediksi benar
- **Classification Report** — Precision, Recall, F1-Score per kelas
- **Confusion Matrix** — Visualisasi prediksi benar vs salah per kelas
- **Cross-Validation** — Estimasi performa yang lebih robust (5-fold)

In [ ]:
# === 5.1 Prediksi ===
y_pred_knn = knn_model.predict(X_test_scaled)
y_pred_rf  = rf_model.predict(X_test_scaled)

acc_knn = accuracy_score(y_test, y_pred_knn)
acc_rf  = accuracy_score(y_test, y_pred_rf)

print('=' * 50)
print(f'📊 KNN Accuracy     : {acc_knn:.4f} ({acc_knn*100:.2f}%)')
print(f'📊 Random Forest Acc: {acc_rf:.4f} ({acc_rf*100:.2f}%)')
print('=' * 50)

In [ ]:
# === 5.2 Classification Report ===
target_names = le.classes_

print('🔹 Classification Report — KNN:')
print(classification_report(y_test, y_pred_knn, target_names=target_names))

print('\n🔹 Classification Report — Random Forest:')
print(classification_report(y_test, y_pred_rf, target_names=target_names))

In [ ]:
# === 5.3 Confusion Matrix ===
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

models_preds = [
    ('KNN', y_pred_knn, 'Blues'),
    ('Random Forest', y_pred_rf, 'Greens')
]

for ax, (name, y_pred, cmap) in zip(axes, models_preds):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
    disp.plot(ax=ax, colorbar=False, cmap=cmap)
    ax.set_title(f'Confusion Matrix — {name}', fontsize=13, fontweight='bold')
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Perbandingan Confusion Matrix', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# === 5.4 Cross-Validation Score (5-Fold) ===
cv_knn = cross_val_score(knn_model, X_train_scaled, y_train, cv=5, scoring='accuracy')
cv_rf  = cross_val_score(rf_model,  X_train_scaled, y_train, cv=5, scoring='accuracy')

print('📊 Cross-Validation Scores (5-Fold):')
print(f'   KNN          : {cv_knn.round(4)} → Mean: {cv_knn.mean():.4f} ± {cv_knn.std():.4f}')
print(f'   Random Forest: {cv_rf.round(4)} → Mean: {cv_rf.mean():.4f} ± {cv_rf.std():.4f}')

In [ ]:
# === 5.5 Visualisasi Perbandingan Model ===
metrics = ['Test Accuracy', 'CV Mean Accuracy']
knn_scores = [acc_knn, cv_knn.mean()]
rf_scores  = [acc_rf,  cv_rf.mean()]

x = np.arange(len(metrics))
width = 0.3

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, knn_scores, width, label='KNN', color='steelblue', edgecolor='black')
bars2 = ax.bar(x + width/2, rf_scores,  width, label='Random Forest', color='seagreen', edgecolor='black')

ax.set_ylim(0.85, 1.05)
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylabel('Accuracy Score')
ax.set_title('Perbandingan Performa KNN vs Random Forest', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

for bar in bars1 + bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.4f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 5), textcoords='offset points',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 📝 Kesimpulan

### Ringkasan Hasil

| Model | Test Accuracy | CV Mean Accuracy |
|---|---|---|
| KNN | ~96–100% | ~96–100% |
| Random Forest | ~96–100% | ~96–100% |

### Insight Utama

1. **Kedua model mencapai akurasi sangat tinggi** (>96%) — ini karena Iris Dataset relatif mudah diklasifikasikan, terutama kelas Iris-setosa yang sangat terpisah dari dua kelas lainnya.

2. **Fitur Petal lebih penting** dibandingkan Sepal — PetalLengthCm dan PetalWidthCm memiliki feature importance tertinggi pada Random Forest dan korelasi yang sangat tinggi (0.96).

3. **Iris-setosa** selalu diklasifikasikan sempurna. Sedikit kesalahan terjadi antara **Iris-versicolor** dan **Iris-virginica** karena distribusinya saling overlap.

4. **Random Forest** cenderung lebih robust karena merupakan ensemble dari banyak decision tree, sehingga lebih tahan terhadap overfitting.

### Rekomendasi
Untuk dataset yang lebih besar dan kompleks, **Random Forest** lebih direkomendasikan karena skalabilitasnya. Untuk dataset kecil dan sederhana seperti Iris, **KNN** sudah cukup efektif dengan preprocessing yang tepat (feature scaling).